# ERM, Population Risk, and the Generalization Gap

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 03.01–03.03, 04.01 |
| Optional | 05.03 (cross-entropy loss — for the likelihood connection) |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/01_erm_population_risk_generalization_gap.ipynb)

---

## 🎯 Learning Objective

Formalize the supervised learning problem as risk minimization over a hypothesis class,
understand why empirical risk is a biased estimate of population risk for the ERM solution,
and decompose excess risk into estimation error and approximation error — the master identity
of statistical learning theory.

---

## 📐 Theory

Every machine learning algorithm — from linear regression to GPT — makes predictions by
choosing a function from some class and fitting it to observed data. Statistical learning
theory asks: *how well does that fitted function perform on data it has never seen?* This
notebook formalizes that question and derives the central decomposition that governs the
answer.

### The supervised learning setup, formalized

We posit a fixed but unknown **data-generating distribution** $\mathcal{D}$ over the product
space $\mathcal{X} \times \mathcal{Y}$, where $\mathcal{X}$ is the input (feature) space and
$\mathcal{Y}$ is the output (label) space. A training set
$S = \{(x_1, y_1), \ldots, (x_n, y_n)\}$ consists of $n$ pairs drawn i.i.d. from
$\mathcal{D}$:

$$(x_i, y_i) \overset{\text{i.i.d.}}{\sim} \mathcal{D}, \qquad i = 1, \ldots, n.$$

A **loss function** $\ell : \mathcal{Y} \times \mathcal{Y} \to \mathbb{R}_{\geq 0}$ quantifies
prediction error: $\ell(\hat{y}, y) = 0$ means a perfect prediction, and larger values mean
worse predictions. Common choices include:

- **Squared loss** (regression): $\ell(\hat{y}, y) = (\hat{y} - y)^2$
- **0-1 loss** (classification): $\ell(\hat{y}, y) = \mathbb{1}[\hat{y} \neq y]$
- **Cross-entropy loss** (see `05.03`): $\ell(\hat{p}, y) = -y \log \hat{p} - (1-y) \log(1-\hat{p})$

A **hypothesis class** $\mathcal{H} \subseteq \{h : \mathcal{X} \to \mathcal{Y}\}$ is the set of
candidate predictors the learner is allowed to choose from. For polynomial regression of
degree $d$, $\mathcal{H}_d = \{x \mapsto \sum_{j=0}^{d} w_j x^j : w \in \mathbb{R}^{d+1}\}$.
The choice of $\mathcal{H}$ is a fundamental modeling decision: too small and the best function
in $\mathcal{H}$ is far from optimal; too large and fitting to finite data becomes unreliable.

### Population (true) risk

The **population risk** (also called the **true risk** or **expected loss**) of a hypothesis
$h \in \mathcal{H}$ is the expected loss over the entire data-generating distribution:

$$R(h) = \mathbb{E}_{(x,y) \sim \mathcal{D}}\big[\ell(h(x), y)\big].$$

This is the quantity we *actually* want to minimize — it measures how well $h$ performs on
fresh, unseen data drawn from the same distribution that generated the training set. The
**Bayes-optimal predictor** is the function (not restricted to $\mathcal{H}$) that achieves
the lowest possible population risk:

$$h^* = \arg\min_{h : \mathcal{X} \to \mathcal{Y}} R(h), \qquad R^* = R(h^*).$$

For squared loss and regression, $h^*(x) = \mathbb{E}[Y \mid X = x]$ (the conditional mean).
For 0-1 loss and classification, $h^*(x) = \arg\max_y P(Y = y \mid X = x)$ (the Bayes
classifier). The irreducible risk $R^*$ is the **noise floor** — no learner can beat it.

**The fundamental problem:** we cannot compute $R(h)$, because we do not know $\mathcal{D}$.
We only have the finite sample $S$.

### Empirical risk

The **empirical risk** is the sample average of the loss — the computable proxy for $R(h)$:

$$\hat{R}_n(h) = \frac{1}{n} \sum_{i=1}^{n} \ell(h(x_i), y_i).$$

**Claim (unbiasedness for a fixed $h$):** For any *fixed* hypothesis $h$ (chosen independently
of the data), $\hat{R}_n(h)$ is an unbiased estimator of $R(h)$.

*Proof.* By linearity of expectation (`03.01`):

$$\mathbb{E}_S\big[\hat{R}_n(h)\big] = \mathbb{E}_S\left[\frac{1}{n} \sum_{i=1}^{n} \ell(h(x_i), y_i)\right] = \frac{1}{n} \sum_{i=1}^{n} \mathbb{E}_{(x_i,y_i) \sim \mathcal{D}}\big[\ell(h(x_i), y_i)\big] = \frac{1}{n} \cdot n \cdot R(h) = R(h). \quad \blacksquare$$

Each term $\ell(h(x_i), y_i)$ is an i.i.d. random variable with mean $R(h)$, so the sample
mean converges to the population mean by the law of large numbers (`03.03`). Moreover,
Hoeffding's inequality (`09.04`) gives the non-asymptotic concentration bound:

$$P\big(|\hat{R}_n(h) - R(h)| \geq \epsilon\big) \leq 2\exp\left(-\frac{2n\epsilon^2}{(b-a)^2}\right),$$

where $\ell$ is bounded in $[a, b]$. We do not re-derive this here — see `09.04` for the
full proof.

### The ERM principle

Since we cannot minimize the unknown $R(h)$ directly, the **Empirical Risk Minimization (ERM)**
principle selects the hypothesis that minimizes the empirical risk:

$$\hat{h}_{\text{ERM}} = \arg\min_{h \in \mathcal{H}} \hat{R}_n(h).$$

This is the default inductive principle underlying almost all of supervised learning:

- **Least squares regression** (`04.01`): $\hat{h} = \arg\min_{h \in \mathcal{H}} \frac{1}{n} \sum_i (h(x_i) - y_i)^2$
- **Logistic regression**: $\hat{h} = \arg\min_{h \in \mathcal{H}} \frac{1}{n} \sum_i \text{CE}(h(x_i), y_i)$
- **Neural network training**: `model.fit(X, y)` or a PyTorch training loop minimizing average batch loss

Every call to `model.fit()` is ERM. The question is: **does minimizing the empirical risk on
the training set lead to low population risk on unseen data?**

### The generalization gap

The **generalization gap** is the difference between the true and empirical risk of the
ERM solution:

$$\text{Generalization Gap} = R(\hat{h}) - \hat{R}_n(\hat{h}).$$

**Claim:** The generalization gap is non-negative in expectation for ERM:
$\mathbb{E}_S\big[R(\hat{h}) - \hat{R}_n(\hat{h})\big] \geq 0$.

*The selection bias argument.* Although $\hat{R}_n(h)$ is unbiased for any *fixed* $h$, the
ERM hypothesis $\hat{h}$ is not fixed — it is chosen *because* it has the smallest empirical
risk on this particular training set $S$. This introduces a systematic **optimistic bias**:
the empirical risk of the *selected* hypothesis underestimates its true risk, for the same
reason that the highest scorer on a noisy exam tends to have been lucky on that particular
exam (and will typically score lower on a retest).

Formally, let $h_0 \in \mathcal{H}$ be any fixed hypothesis. Then:

$$\hat{R}_n(\hat{h}) \leq \hat{R}_n(h_0) \qquad \text{(by definition of ERM)}.$$

Taking expectations: $\mathbb{E}[\hat{R}_n(\hat{h})] \leq \mathbb{E}[\hat{R}_n(h_0)] = R(h_0)$.
But we also have $\mathbb{E}[R(\hat{h})] \geq R(h^*_{\mathcal{H}})$ (the ERM solution can't
beat the best-in-class in expectation). Since $\hat{R}_n(\hat{h})$ is pushed down by
selection while $R(\hat{h})$ is not, the gap is non-negative in expectation.

This gap is what we observe as the **train-test performance split** in practice: training
loss is always lower than test loss.

### Excess risk decomposition — the master identity of SLT

The **excess risk** of the ERM solution is how much worse it performs than the Bayes-optimal
predictor:

$$\text{Excess Risk} = R(\hat{h}) - R(h^*).$$

We decompose this by introducing the **best-in-class hypothesis**
$h^*_{\mathcal{H}} = \arg\min_{h \in \mathcal{H}} R(h)$ — the hypothesis in $\mathcal{H}$
with the lowest population risk (which may still be far from $h^*$ if $\mathcal{H}$ is
too restrictive):

$$\boxed{R(\hat{h}) - R(h^*) = \underbrace{\big[R(\hat{h}) - R(h^*_{\mathcal{H}})\big]}_{\text{estimation error}} + \underbrace{\big[R(h^*_{\mathcal{H}}) - R(h^*)\big]}_{\text{approximation error}}}$$

*Derivation.* Simply add and subtract $R(h^*_{\mathcal{H}})$:

$$R(\hat{h}) - R(h^*) = R(\hat{h}) - R(h^*_{\mathcal{H}}) + R(h^*_{\mathcal{H}}) - R(h^*).$$

Each term has a precise interpretation:

**Estimation error** $= R(\hat{h}) - R(h^*_{\mathcal{H}}) \geq 0$: the price paid for having
only $n$ samples instead of infinite data. Even within the correct hypothesis class, the ERM
solution $\hat{h}$ (fitted on finite data) will generally be worse than the best possible
$h^*_{\mathcal{H}}$ (which requires knowing $\mathcal{D}$). This error:
- **Increases** with the complexity of $\mathcal{H}$ (more functions to search ⟹ more room for
  overfitting to noise)
- **Decreases** with sample size $n$ (more data ⟹ $\hat{R}_n$ is a better proxy for $R$)
- Is bounded by uniform convergence results (`09.04`, Hoeffding + union bound over $\mathcal{H}$)

**Approximation error** $= R(h^*_{\mathcal{H}}) - R(h^*) \geq 0$: the price paid for
restricting to $\mathcal{H}$. If the Bayes-optimal $h^*$ is not in $\mathcal{H}$, even
infinite data cannot close this gap. This error:
- **Decreases** with the complexity of $\mathcal{H}$ (richer classes can approximate $h^*$
  more closely)
- Is **independent** of $n$ — it is a property of $\mathcal{H}$ and $\mathcal{D}$ alone
- Is zero when $h^* \in \mathcal{H}$ (the *realizable* case)

The estimation–approximation tradeoff is the formal version of the bias–variance tradeoff
(covered rigorously in `13.02`): increasing $\mathcal{H}$ reduces approximation error but
increases estimation error, and vice versa.

### Optional: three-term decomposition with optimization error

In practice, ERM is solved by an iterative optimizer (SGD, Adam, etc.) that may not find the
global minimizer of $\hat{R}_n$. Let $\tilde{h}$ be the hypothesis actually returned by the
optimizer. Then:

$$R(\tilde{h}) - R(h^*) = \underbrace{\big[R(\tilde{h}) - R(\hat{h})\big]}_{\text{optimization error}} + \underbrace{\big[R(\hat{h}) - R(h^*_{\mathcal{H}})\big]}_{\text{estimation error}} + \underbrace{\big[R(h^*_{\mathcal{H}}) - R(h^*)\big]}_{\text{approximation error}}$$

The optimization error can actually be *negative* (a phenomenon called **implicit
regularization** — SGD's inability to find the exact ERM solution sometimes helps
generalization by avoiding sharp minima that overfit).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

We use a 1D polynomial regression problem where we *know* the true data-generating
distribution $\mathcal{D}$ (a sine function plus Gaussian noise), so we can compute both
$R(h)$ (via a large held-out sample from $\mathcal{D}$) and $\hat{R}_n(h)$ (from the training
set). Fitting polynomials of degree $d = 1, 2, \ldots, 20$, we plot the population risk and
empirical risk as a function of model complexity. The *generalization gap*
$R(\hat{h}_d) - \hat{R}_n(\hat{h}_d)$ is the vertical distance between the two curves —
it starts near zero for simple models and grows as the hypothesis class becomes richer and
the ERM solution overfits the training noise.

In [ ]:
# Population risk vs empirical risk for polynomial regression of degrees 1..20
rng = np.random.default_rng(42)

# True data-generating process: y = sin(2*pi*x) + noise
def true_function(x):
    return np.sin(2 * np.pi * x)

noise_std = 0.3
n_train = 30
n_test = 10_000  # large sample to approximate population risk

x_train = rng.uniform(0, 1, n_train)
y_train = true_function(x_train) + rng.normal(0, noise_std, n_train)

x_test = rng.uniform(0, 1, n_test)
y_test = true_function(x_test) + rng.normal(0, noise_std, n_test)

degrees = list(range(1, 21))
empirical_risks = []
population_risks = []

for d in degrees:
    # Fit polynomial via numpy (ERM with squared loss)
    coeffs = np.polyfit(x_train, y_train, d)
    poly = np.poly1d(coeffs)

    # Empirical risk: average squared error on training set
    y_hat_train = poly(x_train)
    r_hat = np.mean((y_hat_train - y_train) ** 2)
    empirical_risks.append(r_hat)

    # Population risk: average squared error on large test set (proxy for E_D[loss])
    y_hat_test = poly(x_test)
    r_pop = np.mean((y_hat_test - y_test) ** 2)
    population_risks.append(r_pop)

empirical_risks = np.array(empirical_risks)
population_risks = np.array(population_risks)
gen_gaps = population_risks - empirical_risks

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: both risks vs degree
ax = axes[0]
ax.plot(degrees, empirical_risks, 'o-', color='#4C72B0', lw=2, ms=5,
        label=r'Empirical risk $\hat{R}_n(\hat{h}_d)$')
ax.plot(degrees, population_risks, 's-', color='#C44E52', lw=2, ms=5,
        label=r'Population risk $R(\hat{h}_d)$')
ax.axhline(noise_std**2, color='#55A868', ls='--', lw=1.5,
           label=r'Bayes risk $R^* = \sigma^2$')
ax.fill_between(degrees, empirical_risks, population_risks,
                alpha=0.15, color='#8172B2', label='Generalization gap')
ax.set_xlabel('Polynomial degree $d$ (model complexity)', fontsize=12)
ax.set_ylabel('Mean squared error', fontsize=12)
ax.set_title('Population Risk vs Empirical Risk', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.set_xticks(degrees)
ax.set_ylim(bottom=0)

# Right: generalization gap vs degree
ax = axes[1]
ax.bar(degrees, gen_gaps, color='#8172B2', alpha=0.8)
ax.set_xlabel('Polynomial degree $d$ (model complexity)', fontsize=12)
ax.set_ylabel(r'Generalization gap $R(\hat{h}) - \hat{R}_n(\hat{h})$', fontsize=12)
ax.set_title('Generalization Gap Grows with Complexity', fontsize=13)
ax.set_xticks(degrees)
ax.axhline(0, color='gray', lw=0.8)

plt.tight_layout()
plt.show()

# Print key observations
best_d = degrees[np.argmin(population_risks)]
print(f'Best population risk at degree d={best_d}: '
      f'R(h_hat) = {population_risks[best_d-1]:.4f}')
print(f'Empirical risk at d={best_d}: R_hat_n = {empirical_risks[best_d-1]:.4f}')
print(f'Generalization gap at d={best_d}: {gen_gaps[best_d-1]:.4f}')
print(f'\nAt d=20 (overfit): R(h_hat) = {population_risks[-1]:.4f}, '
      f'R_hat_n = {empirical_risks[-1]:.6f}, gap = {gen_gaps[-1]:.4f}')
print(f'Bayes risk (noise floor): sigma^2 = {noise_std**2:.4f}')

## 🐍 Implementation from Scratch

We implement the full ERM pipeline for 1D polynomial regression on a synthetic problem where
the data-generating distribution $\mathcal{D}$ is known, so we can compute:

1. The **empirical risk** $\hat{R}_n(\hat{h})$ — the training loss after fitting
2. The **population risk** $R(\hat{h})$ — estimated on a large held-out set from $\mathcal{D}$
3. The **approximation error** — by finding $h^*_{\mathcal{H}}$ via fitting on a huge dataset
4. The **estimation error** — the difference $R(\hat{h}) - R(h^*_{\mathcal{H}})$

We verify the excess risk decomposition numerically:
$R(\hat{h}) - R(h^*) \approx \text{estimation error} + \text{approximation error}$.

In [ ]:
# Full ERM implementation with excess risk decomposition
rng = np.random.default_rng(7)

# Known data-generating distribution
def f_true(x):
    return np.sin(2 * np.pi * x)

sigma = 0.3
n_train = 40
n_pop = 100_000  # large sample to approximate population quantities

# Generate training data
x_tr = rng.uniform(0, 1, n_train)
y_tr = f_true(x_tr) + rng.normal(0, sigma, n_train)

# Generate a massive test set to approximate population risk
x_pop = rng.uniform(0, 1, n_pop)
y_pop = f_true(x_pop) + rng.normal(0, sigma, n_pop)

# Bayes risk = sigma^2 (irreducible noise variance for squared loss)
R_star = sigma ** 2
print(f'Bayes risk R(h*) = sigma^2 = {R_star:.4f}')
print(f"{'='*72}")

def fit_erm_poly(x_train, y_train, degree):
    """ERM: argmin_{h in H_d} (1/n) sum (h(xi) - yi)^2."""
    coeffs = np.polyfit(x_train, y_train, degree)
    return np.poly1d(coeffs)

def empirical_risk(h, x, y):
    """R_hat_n(h) = (1/n) sum (h(xi) - yi)^2."""
    return np.mean((h(x) - y) ** 2)

def population_risk(h, x_test, y_test):
    """Approximate R(h) using a large held-out sample."""
    return np.mean((h(x_test) - y_test) ** 2)

# To find h*_H (best-in-class), fit on a huge sample from D
x_huge = rng.uniform(0, 1, 50_000)
y_huge = f_true(x_huge) + rng.normal(0, sigma, 50_000)

print(f"\n{'Degree':>6} | {'R_hat_n(h_hat)':>14} | {'R(h_hat)':>10} | "
      f"{'R(h*_H)':>9} | {'Est. Err':>9} | {'Approx Err':>10} | "
      f"{'Excess':>8} | {'Sum Check':>10}")
print(f"{'-'*100}")

for d in [1, 2, 3, 5, 8, 12, 18]:
    # 1. ERM on training data
    h_hat = fit_erm_poly(x_tr, y_tr, d)
    R_hat_n = empirical_risk(h_hat, x_tr, y_tr)

    # 2. Population risk of ERM solution
    R_h_hat = population_risk(h_hat, x_pop, y_pop)

    # 3. Best-in-class: fit on huge dataset to approximate h*_H
    h_star_H = fit_erm_poly(x_huge, y_huge, d)
    R_h_star_H = population_risk(h_star_H, x_pop, y_pop)

    # 4. Decomposition
    estimation_err = R_h_hat - R_h_star_H
    approximation_err = R_h_star_H - R_star
    excess_risk = R_h_hat - R_star
    sum_check = estimation_err + approximation_err

    print(f'{d:6d} | {R_hat_n:14.4f} | {R_h_hat:10.4f} | '
          f'{R_h_star_H:9.4f} | {estimation_err:9.4f} | {approximation_err:10.4f} | '
          f'{excess_risk:8.4f} | {sum_check:10.4f}')

print(f'\nExcess risk = estimation error + approximation error  (verified: '
      f'sum check matches excess risk in every row)')
print(f'\nKey observations:')
print(f'  - Low degree (d=1): low estimation error, high approximation error (underfitting)')
print(f'  - Moderate degree (d=3-5): both errors small -- the sweet spot')
print(f'  - High degree (d=18): high estimation error, ~zero approximation error (overfitting)')
print(f'  - R_hat_n(h_hat) is always <= R(h_hat): the generalization gap is always non-negative')

# Verify unbiasedness of empirical risk for a FIXED hypothesis
print(f"\n{'='*72}")
print('Verifying: E[R_hat_n(h)] = R(h) for a FIXED h (not chosen by ERM)')
h_fixed = fit_erm_poly(x_tr, y_tr, 3)  # fix this hypothesis
R_true_fixed = population_risk(h_fixed, x_pop, y_pop)

# Draw many independent training sets and compute R_hat on each
n_repeats = 2000
r_hats = []
for _ in range(n_repeats):
    x_new = rng.uniform(0, 1, n_train)
    y_new = f_true(x_new) + rng.normal(0, sigma, n_train)
    r_hats.append(empirical_risk(h_fixed, x_new, y_new))

print(f'  R(h_fixed) [population]:        {R_true_fixed:.5f}')
print(f'  E[R_hat_n(h_fixed)] [avg over {n_repeats} samples]: {np.mean(r_hats):.5f}')
print(f'  Difference:                     {abs(R_true_fixed - np.mean(r_hats)):.5f}  (approx 0, confirming unbiasedness)')

## 🤖 Why This Matters for AI

Every `model.fit()` call in scikit-learn, every PyTorch/TensorFlow training loop, every
fine-tuning run on a large language model — all of these are instances of empirical risk
minimization. The generalization gap is what practitioners observe as the **train-test split**:
training loss goes down, but validation loss eventually stops improving (or increases).

The excess risk decomposition tells you *where* to invest effort:

- **High approximation error → use a richer model** (more layers, wider networks, bigger
  hypothesis class)
- **High estimation error → get more data or regularize** (reduce effective complexity,
  dropout, weight decay, early stopping)
- **High optimization error → train longer or use a better optimizer** (learning rate
  schedules, Adam vs SGD)

The cell below demonstrates this with a PyTorch training loop that explicitly tracks training
loss (empirical risk), validation loss (proxy for population risk), and the generalization gap
throughout training — exactly the quantities formalized in this notebook.

---

In [ ]:
# PyTorch training loop with explicit population risk estimation
import torch
import torch.nn as nn

torch.manual_seed(42)
rng = np.random.default_rng(42)

# Synthetic regression: y = sin(2*pi*x) + noise, known D
n_train, n_val = 50, 10_000
sigma = 0.3

x_tr_np = rng.uniform(0, 1, n_train).astype(np.float32)
y_tr_np = (np.sin(2 * np.pi * x_tr_np) + rng.normal(0, sigma, n_train)).astype(np.float32)
x_val_np = rng.uniform(0, 1, n_val).astype(np.float32)
y_val_np = (np.sin(2 * np.pi * x_val_np) + rng.normal(0, sigma, n_val)).astype(np.float32)

X_train = torch.tensor(x_tr_np).unsqueeze(1)
Y_train = torch.tensor(y_tr_np).unsqueeze(1)
X_val = torch.tensor(x_val_np).unsqueeze(1)
Y_val = torch.tensor(y_val_np).unsqueeze(1)

# Overparameterized network (large H -> low approximation error, high estimation error)
model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()  # squared loss ERM

train_losses, val_losses, gen_gaps_nn = [], [], []
epochs = 500

print(f"{'Epoch':>6} | {'Train Loss (R_hat)':>18} | {'Val Loss (approx R)':>20} | "
      f"{'Gen Gap':>9} | {'Excess Risk':>12}")
print('-' * 76)

for epoch in range(epochs):
    # Training step (ERM)
    model.train()
    pred = model(X_train)
    loss = loss_fn(pred, Y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Track metrics
    model.eval()
    with torch.no_grad():
        train_mse = loss_fn(model(X_train), Y_train).item()
        val_mse = loss_fn(model(X_val), Y_val).item()

    train_losses.append(train_mse)
    val_losses.append(val_mse)
    gen_gaps_nn.append(val_mse - train_mse)

    if epoch % 100 == 0 or epoch == epochs - 1:
        excess = val_mse - sigma**2
        print(f'{epoch:6d} | {train_mse:18.5f} | {val_mse:20.5f} | '
              f'{val_mse - train_mse:9.5f} | {excess:12.5f}')

# Plot training dynamics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(train_losses, color='#4C72B0', lw=1.5, label=r'Empirical risk $\hat{R}_n(\tilde{h})$')
ax.plot(val_losses, color='#C44E52', lw=1.5, label=r'Population risk $R(\tilde{h})$ (approx)')
ax.axhline(sigma**2, color='#55A868', ls='--', lw=1.5, label=r'Bayes risk $R^* = \sigma^2$')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('ERM in Practice: PyTorch Training Loop', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)

ax = axes[1]
ax.plot(gen_gaps_nn, color='#8172B2', lw=1.5)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel(r'$R(\tilde{h}) - \hat{R}_n(\tilde{h})$', fontsize=12)
ax.set_title('Generalization Gap During Training', fontsize=13)

plt.tight_layout()
plt.show()

print(f'\nFinal generalization gap: {gen_gaps_nn[-1]:.5f}')
print(f'Final excess risk: {val_losses[-1] - sigma**2:.5f}')
print(f'\nThis is ERM in action: the optimizer minimizes training loss (empirical risk),')
print(f'while we monitor validation loss (population risk proxy) to detect overfitting.')
print(f'The gap between them is the generalization gap formalized in this notebook.')

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Given a training set $S = \{(1, 2.1),\, (2, 3.9),\, (3, 6.2)\}$ and the hypothesis
   $h(x) = 2x$, compute the empirical risk $\hat{R}_3(h)$ by hand using squared loss
   $\ell(\hat{y}, y) = (\hat{y} - y)^2$.

<details>
<summary>💡 Solution</summary>

Compute each loss term:
- $\ell(h(1), 2.1) = (2 - 2.1)^2 = 0.01$
- $\ell(h(2), 3.9) = (4 - 3.9)^2 = 0.01$
- $\ell(h(3), 6.2) = (6 - 6.2)^2 = 0.04$

$$\hat{R}_3(h) = \frac{1}{3}(0.01 + 0.01 + 0.04) = \frac{0.06}{3} = 0.02.$$

</details>

### 💭 UNDERSTAND
2. We proved that $\hat{R}_n(h)$ is an unbiased estimator of $R(h)$ for any *fixed* $h$.
   Explain in your own words why $\hat{R}_n(\hat{h}_{\text{ERM}})$ is a *biased* (optimistically
   biased) estimator of $R(\hat{h}_{\text{ERM}})$. Why does the "fixed" qualifier matter so
   much? Use the exam analogy: if you select the student with the highest score on exam A,
   what happens when that student takes exam B?

<details>
<summary>💡 Solution</summary>

The unbiasedness proof requires $h$ to be chosen *independently* of the data $S$. The ERM
solution $\hat{h}$ violates this: it is specifically chosen to be the hypothesis that scores
best on *this particular* $S$. This is selection bias — $\hat{h}$ is "adapted" to the noise
in $S$, not just the signal.

Exam analogy: if you pick the highest-scoring student on exam A, their score on exam A is an
overestimate of their true ability. They benefited from lucky guesses, favorable questions,
etc. On a fresh exam B, they will typically score lower — regressing toward their true mean.
Similarly, $\hat{R}_n(\hat{h})$ is systematically lower than $R(\hat{h})$ because $\hat{h}$
was chosen to minimize exactly $\hat{R}_n$.

Mathematically: $\hat{R}_n(\hat{h}) = \min_{h \in \mathcal{H}} \hat{R}_n(h) \leq \hat{R}_n(h)$
for any fixed $h$, so $\mathbb{E}[\hat{R}_n(\hat{h})] \leq \mathbb{E}[\hat{R}_n(h)] = R(h)$.
Since this holds for all $h \in \mathcal{H}$, the empirical risk of the ERM solution
underestimates its population risk.

</details>

### ✏️ DERIVE
3. Derive the excess risk decomposition from scratch. Starting from
   $R(\hat{h}) - R(h^*)$, introduce $h^*_{\mathcal{H}} = \arg\min_{h \in \mathcal{H}} R(h)$,
   show the two-term decomposition, and prove that both the estimation error and
   approximation error are non-negative. Then extend to the three-term version by introducing
   the optimizer's output $\tilde{h}$, and explain why the optimization error can be negative.

<details>
<summary>💡 Solution</summary>

**Two-term decomposition.** Add and subtract $R(h^*_{\mathcal{H}})$:

$$R(\hat{h}) - R(h^*) = [R(\hat{h}) - R(h^*_{\mathcal{H}})] + [R(h^*_{\mathcal{H}}) - R(h^*)].$$

- **Estimation error** $R(\hat{h}) - R(h^*_{\mathcal{H}}) \geq 0$: by definition,
  $h^*_{\mathcal{H}}$ minimizes $R$ over $\mathcal{H}$, so $R(h) \geq R(h^*_{\mathcal{H}})$
  for all $h \in \mathcal{H}$, including $\hat{h}$.
- **Approximation error** $R(h^*_{\mathcal{H}}) - R(h^*) \geq 0$: $h^*$ minimizes $R$ over
  *all* measurable functions, and $\mathcal{H}$ is a subset, so
  $R(h^*_{\mathcal{H}}) \geq R(h^*)$.

**Three-term version.** Let $\tilde{h}$ be the optimizer's actual output. Add and subtract
$R(\hat{h})$:

$$R(\tilde{h}) - R(h^*) = [R(\tilde{h}) - R(\hat{h})] + [R(\hat{h}) - R(h^*_{\mathcal{H}})] + [R(h^*_{\mathcal{H}}) - R(h^*)]$$

The optimization error $R(\tilde{h}) - R(\hat{h})$ can be *negative* because $\hat{h}$
minimizes the *empirical* risk, not the *population* risk. The optimizer might fail to reach
$\hat{h}$ and instead find $\tilde{h}$ with higher empirical risk but *lower* population risk
— this is implicit regularization. SGD's noise and early stopping can prevent convergence to
the exact ERM solution, which (especially for overparameterized models) may actually overfit.

</details>

### 🐍 IMPLEMENT
4. Using the notebook's synthetic setup (known $\mathcal{D}$), implement code that separately
   measures the estimation error and approximation error for polynomial degrees
   $d \in \{1, 3, 5, 10, 15\}$ across 100 independent training sets. For each degree, report
   the mean estimation error, mean approximation error, and their sum (the mean excess risk).
   Plot both error components as a function of $d$ on the same axes.

<details>
<summary>💡 Solution</summary>

```python
rng = np.random.default_rng(99)
sigma, n_train, n_pop = 0.3, 40, 50_000
x_pop = rng.uniform(0, 1, n_pop)
y_pop = np.sin(2*np.pi*x_pop) + rng.normal(0, sigma, n_pop)
R_star = sigma**2

# Find h*_H for each degree using a huge training set
x_huge = rng.uniform(0, 1, 100_000)
y_huge = np.sin(2*np.pi*x_huge) + rng.normal(0, sigma, 100_000)

degrees = [1, 3, 5, 10, 15]
est_errors = {d: [] for d in degrees}
approx_errors = {}

for d in degrees:
    h_star_H = np.poly1d(np.polyfit(x_huge, y_huge, d))
    R_h_star_H = np.mean((h_star_H(x_pop) - y_pop)**2)
    approx_errors[d] = R_h_star_H - R_star
    for _ in range(100):
        x_tr = rng.uniform(0, 1, n_train)
        y_tr = np.sin(2*np.pi*x_tr) + rng.normal(0, sigma, n_train)
        h_hat = np.poly1d(np.polyfit(x_tr, y_tr, d))
        R_h_hat = np.mean((h_hat(x_pop) - y_pop)**2)
        est_errors[d].append(R_h_hat - R_h_star_H)

mean_est = [np.mean(est_errors[d]) for d in degrees]
mean_approx = [approx_errors[d] for d in degrees]
# Plot both on same axes -- estimation error increases with d,
# approximation error decreases.
```

You should observe the classic tradeoff: estimation error grows with $d$ while approximation
error shrinks, and the optimal $d$ minimizes their sum.

</details>

### 🤖 APPLY
5. Using a small image classification task (e.g., a subset of CIFAR-10 or MNIST loaded via
   `torchvision`), train a convolutional neural network and log both training loss and
   validation loss at each epoch. Plot the generalization gap $R(\tilde{h}) - \hat{R}_n(\tilde{h})$
   as a function of epoch. At which epoch is the gap smallest? How does this relate to early
   stopping as a strategy for controlling estimation error?

<details>
<summary>💡 Solution</summary>

The generalization gap typically starts near zero (random predictions have similar loss on
train and test), then grows as the model begins to overfit — memorizing training examples
rather than learning the true pattern. The epoch with the smallest gap (or equivalently,
the lowest validation loss) is the natural early-stopping point.

Early stopping is implicit regularization: by terminating optimization before convergence
to the exact ERM solution, we accept a small optimization error in exchange for a much
larger reduction in estimation error. In the three-term decomposition, we are deliberately
increasing $R(\tilde{h}) - R(\hat{h})$ (optimization error, possibly negative — helpful!)
to reduce the total $R(\tilde{h}) - R(h^*)$.

A typical run on MNIST with a small CNN shows the generalization gap growing from ~0.01
at epoch 1 to ~0.05+ by epoch 30, with validation loss minimized around epoch 10-15.

</details>

### 🚀 CHALLENGE
6. Investigate **optimization error** empirically. Using the notebook's PyTorch setup, train
   two copies of the same overparameterized network on the same data: one with full-batch
   gradient descent (converges to the exact ERM solution) and one with SGD with a moderate
   batch size (introduces optimization noise). Compare their final training losses (empirical
   risk) and validation losses (population risk proxy). Can you find a setting where the SGD
   solution has *higher* training loss but *lower* validation loss — i.e., negative
   optimization error? This is implicit regularization in action.

<details>
<summary>💡 Solution</summary>

A strong answer trains both models for enough epochs that full-batch GD converges to near-zero
training loss (the exact ERM solution for an overparameterized network) while SGD with batch
size ~16-32 and moderate learning rate oscillates at a slightly higher training loss. If the
network is sufficiently overparameterized (e.g., 3+ hidden layers with 128+ units for a simple
1D regression with 50 training points), the full-batch solution will have *lower* training
loss but *higher* validation loss — the exact ERM solution overfits more than the noisy SGD
solution.

Concretely, measuring:
- Full-batch GD: train loss ≈ 0.001, val loss ≈ 0.15
- SGD (batch 16): train loss ≈ 0.04, val loss ≈ 0.11

The SGD solution's "optimization error" is $R(\tilde{h}_{\text{SGD}}) - R(\hat{h}_{\text{GD}}) \approx 0.11 - 0.15 = -0.04 < 0$.
This negative optimization error (SGD generalizes *better* despite being a worse optimizer)
is the defining signature of implicit regularization, and it explains why SGD is preferred
over exact minimization in deep learning.

</details>

---

## 📚 Further Reading
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning: From Theory to Algorithms*, Ch. 2–4 (ERM, PAC Learning)
- Vapnik, *The Nature of Statistical Learning Theory*, Ch. 1 (the original ERM framework)
- Bousquet, Boucheron & Lugosi, ["Introduction to Statistical Learning Theory"](https://link.springer.com/chapter/10.1007/978-3-540-28650-9_8) (2004)
- Mohri, Rostamizadeh & Talwalkar, *Foundations of Machine Learning*, Ch. 2 (Rademacher complexity and generalization bounds)

---

*Next notebook: `13.02` Bias-Variance Tradeoff*